# TurboMemory Live Demo

Claude-style long-term memory with 4/6/8-bit TurboQuant compression — runs on a laptop.

[![GitHub](https://img.shields.io/badge/GitHub-Kubenew/TurboMemory-blue)](https://github.com/Kubenew/TurboMemory)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)]()

In [ ]:
# Install TurboMemory
!pip install -q git+https://github.com/Kubenew/TurboMemory.git sentence-transformers

## 1. Initialize TurboMemory

In [ ]:
from turbomemory import TurboMemory

tm = TurboMemory(root="demo_memory")
print("TurboMemory initialized!")

## 2. Add Memories

In [ ]:
# Add some memories with different quantization levels
tm.add_memory("python", "Python is a high-level programming language created by Guido van Rossum in 1991", bits=6)
tm.add_memory("python", "Python uses dynamic typing and garbage collection", bits=6)
tm.add_memory("rust", "Rust is a systems programming language focused on memory safety without garbage collection", bits=4)
tm.add_memory("rust", "Rust uses ownership and borrowing to prevent data races at compile time", bits=4)
tm.add_memory("ml", "Transformers use self-attention mechanisms for sequence processing", bits=8)
tm.add_memory("ml", "Quantization reduces model size by representing weights with fewer bits", bits=8)

print("Added 6 memories across 3 topics")

## 3. Query Memory

In [ ]:
results = tm.query("programming languages", k=3)

for score, topic, chunk in results:
    print(f"\nScore: {score:.4f} | Topic: {topic}")
    print(f"Text: {chunk['text']}")
    print(f"Quality: {chunk.get('quality_score', 0):.3f}")

## 4. Query with Verification

In [ ]:
results = tm.verify_and_score("memory safety in programming", k=3)

for score, topic, chunk, verif in results:
    status = "VERIFIED" if verif.verified else "UNVERIFIED"
    print(f"\n{status} | Score: {score:.4f} | Topic: {topic}")
    print(f"Text: {chunk['text']}")
    print(f"Cross-refs: {verif.cross_references} | Agreement: {verif.agreement_score:.3f}")

## 5. View Stats & Topic Health

In [ ]:
import json
print(json.dumps(tm.stats(), indent=2))

## 6. Exclusion Rules in Action

In [ ]:
# These should be excluded by default rules
result1 = tm.add_memory("debug", "Error: stack trace at line 42 in module.py")
result2 = tm.add_memory("secrets", "password = my_secret_password_123")

print(f"Debug output stored: {result1 is not None}")  # Should be False
print(f"Secret stored: {result2 is not None}")  # Should be False

## 7. Compression Comparison

In [ ]:
import numpy as np
from turbomemory import quantize_packed, dequantize_packed, cosine_sim

# Create a sample embedding
vec = np.random.randn(384).astype(np.float32)

print("Compression Comparison (384-dim vector):")
print(f"{'Bits':<6} {'Original (bytes)':<18} {'Compressed (bytes)':<20} {'Ratio':<8} {'Similarity':<12}")
print("-" * 70)

original_size = 384 * 4  # float32

for bits in [4, 6, 8]:
    q = quantize_packed(vec, bits=bits)
    import base64
    compressed = len(base64.b64decode(q["data"]))
    reconstructed = dequantize_packed(q)
    sim = cosine_sim(vec, reconstructed)
    ratio = compressed / original_size
    print(f"{bits:<6} {original_size:<18} {compressed:<20} {ratio:<8.2f}x {sim:<12.4f}")

## 8. LangChain Integration

In [ ]:
# Install LangChain if needed
# !pip install -q langchain langchain-core

from turbomemory.integrations.langchain import TurboMemoryRetriever

retriever = TurboMemoryRetriever(root="demo_memory", k=3)
docs = retriever.invoke("What is quantization?")

for doc in docs:
    print(f"\nTopic: {doc.metadata['topic']} | Score: {doc.metadata['score']:.3f}")
    print(f"Content: {doc.page_content[:100]}...")

---

**Next Steps:**
- [GitHub Repository](https://github.com/Kubenew/TurboMemory)
- [Documentation](https://github.com/Kubenew/TurboMemory#readme)
- [Report Issues](https://github.com/Kubenew/TurboMemory/issues)
- [Contribute](https://github.com/Kubenew/TurboMemory/blob/main/CONTRIBUTING.md)